In [0]:
# ============================================================
# PFIN | Phase 2 | Silver Transformation
# Notebook:  02_transform_silver
# Source:    pfin_dev.bronze.elections_canada_contributions_raw
# Target:    pfin_dev.silver.{recipients, contributors,
#            electoral_events, contributions}
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import DateType, DecimalType, IntegerType
from datetime import datetime

# ── CONFIG ──────────────────────────────────────────────────
TARGET_CATALOG  = "pfin_dev"
TARGET_SCHEMA   = "silver"
SOURCE_TABLE    = f"{TARGET_CATALOG}.bronze.elections_canada_contributions_raw"

TABLES = {
    "recipients":       f"{TARGET_CATALOG}.{TARGET_SCHEMA}.recipients",
    "contributors":     f"{TARGET_CATALOG}.{TARGET_SCHEMA}.contributors",
    "electoral_events": f"{TARGET_CATALOG}.{TARGET_SCHEMA}.electoral_events",
    "contributions":    f"{TARGET_CATALOG}.{TARGET_SCHEMA}.contributions",
}

# ============================================================
# STEP 1: READ BRONZE + DEDUP
# ============================================================
print(f"[{datetime.now()}] Reading Bronze...")
df_raw = spark.table(SOURCE_TABLE)
total_raw = df_raw.count()
print(f"  Bronze rows: {total_raw:,}")

src_cols = [c for c in df_raw.columns if c not in ("_source_file", "_ingested_at")]
df = df_raw.dropDuplicates(src_cols)
deduped = df.count()
print(f"  After dedup:  {deduped:,}  (dropped {total_raw - deduped:,})")

# ============================================================
# STEP 2: FIX ENCODING (double/triple-encoded UTF-8)
# ============================================================
# Source CSVs contain French characters that were double-encoded
# through ISO-8859-1 → UTF-8 round-trips. Fix by reversing:
#   encode(col, "ISO-8859-1") → decode(result, "UTF-8")
# Two passes catch both double and triple encoding.
# Final pass strips orphan Â characters (garbage residue).

text_cols_to_fix = [
    "political_entity", "recipient", "recipient_last_name",
    "recipient_first_name", "recipient_middle_initial",
    "political_party_of_recipient", "electoral_district",
    "electoral_event", "contributor_name", "contributor_last_name",
    "contributor_first_name", "contributor_middle_initial",
    "contributor_city", "contributor_postal_code",
    "leadership_contestant", "financial_report",
]

for col_name in text_cols_to_fix:
    c = F.col(col_name)
    # Pass 1
    df = df.withColumn(col_name,
        F.when(c.contains("Ã"), F.decode(F.encode(c, "ISO-8859-1"), "UTF-8")).otherwise(c))
    # Pass 2
    c2 = F.col(col_name)
    df = df.withColumn(col_name,
        F.when(c2.contains("Ã"), F.decode(F.encode(c2, "ISO-8859-1"), "UTF-8")).otherwise(c2))
    # Pass 3: strip orphan Â
    df = df.withColumn(col_name, F.regexp_replace(F.col(col_name), "Â", ""))
    df = df.withColumn(col_name, F.trim(F.col(col_name)))

# ── Strip BOM prefix from political_entity ──────────────────
df = df.withColumn("political_entity",
    F.regexp_replace("political_entity", "^ï»¿", ""))

print(f"[{datetime.now()}] Encoding fixes applied.")

# ============================================================
# STEP 3: STANDARDIZE PROVINCE
# ============================================================
prov_clean = F.regexp_replace(F.upper(F.trim(F.col("contributor_province"))), r"\.$", "")

df = df.withColumn("contributor_province",
    F.when(prov_clean.isin("AB", "ALBERTA"), "AB")
     .when(prov_clean.isin("BC", "BRITISH COLUMBIA"), "BC")
     .when(prov_clean.isin("MB"), "MB")
     .when(prov_clean.isin("NB"), "NB")
     .when(prov_clean.isin("NL"), "NL")
     .when(prov_clean.isin("NS"), "NS")
     .when(prov_clean.isin("NT"), "NT")
     .when(prov_clean.isin("NU"), "NU")
     .when(prov_clean.isin("ON", "ONT", "ONTARIO"), "ON")
     .when(prov_clean.isin("PE", "PEI"), "PE")
     .when(prov_clean.rlike("(?i)(^QC$|^AQC$|QU|BEC)"), "QC")
     .when(prov_clean.isin("SK", "SASKATCHEWAN"), "SK")
     .when(prov_clean.isin("YT"), "YT")
     .otherwise(None)
)

# ============================================================
# STEP 4: CLEAN TEXT FIELDS
# ============================================================
# Postal code: uppercase, no spaces
df = df.withColumn("contributor_postal_code",
    F.upper(F.regexp_replace(F.trim(F.col("contributor_postal_code")), r"\s+", "")))

# City: uppercase, trimmed
df = df.withColumn("contributor_city",
    F.upper(F.trim(F.col("contributor_city"))))

# Nullify "None" strings
for col_name in ["electoral_event", "contribution_given_through", "leadership_contestant"]:
    df = df.withColumn(col_name,
        F.when(F.trim(F.col(col_name)).isin("None", ""), None).otherwise(F.col(col_name)))

# ============================================================
# STEP 5: TYPE CASTING
# ============================================================
# Fix "0025-*" → "2025-*" date typo
df = df.withColumn("contribution_received_date",
    F.regexp_replace("contribution_received_date", "^0025-", "2025-"))

# Dates
df = df.withColumn("fiscal_election_date", F.col("fiscal_election_date").cast(DateType()))
df = df.withColumn("contribution_received_date", F.col("contribution_received_date").cast(DateType()))

# Monetary
df = df.withColumn("monetary_amount", F.trim(F.col("monetary_amount")).cast(DecimalType(12, 2)))
df = df.withColumn("non_monetary_amount", F.trim(F.col("non_monetary_amount")).cast(DecimalType(12, 2)))
df = df.withColumn("total_amount",
    F.coalesce(F.col("monetary_amount"), F.lit(0).cast(DecimalType(12, 2))) +
    F.coalesce(F.col("non_monetary_amount"), F.lit(0).cast(DecimalType(12, 2))))

# IDs and derived fields
df = df.withColumn("recipient_id", F.col("recipient_id").cast(IntegerType()))
df = df.withColumn("fiscal_year", F.year("fiscal_election_date"))

# ============================================================
# STEP 6: GENERATE SURROGATE KEYS
# ============================================================
df = df.withColumn("contributor_key",
    F.sha2(F.concat_ws("|",
        F.upper(F.trim(F.coalesce(F.col("contributor_last_name"), F.lit("")))),
        F.upper(F.trim(F.coalesce(F.col("contributor_first_name"), F.lit("")))),
        F.upper(F.trim(F.coalesce(F.col("contributor_postal_code"), F.lit(""))))
    ), 256))

df = df.withColumn("electoral_event_key",
    F.sha2(F.concat_ws("|",
        F.upper(F.trim(F.coalesce(F.col("electoral_event"), F.lit("")))),
        F.coalesce(F.col("fiscal_election_date").cast("string"), F.lit(""))
    ), 256))

df = df.withColumn("_transformed_at", F.current_timestamp())

print(f"[{datetime.now()}] Cleaning and type casting complete.")

# ============================================================
# STEP 7: WRITE SILVER TABLES
# ============================================================

# ── recipients ──────────────────────────────────────────────
print(f"[{datetime.now()}] Writing recipients...")
df_recipients = (
    df.select(
        "recipient_id",
        F.col("recipient").alias("recipient_name"),
        "recipient_last_name",
        "recipient_first_name",
        "recipient_middle_initial",
        "political_entity",
        F.col("political_party_of_recipient").alias("political_party"),
        "electoral_district",
    )
    .dropDuplicates(["recipient_id"])
)
df_recipients.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLES["recipients"])
rc = spark.table(TABLES["recipients"]).count()
print(f"  recipients: {rc:,} rows")

# ── contributors ────────────────────────────────────────────
print(f"[{datetime.now()}] Writing contributors...")
df_contributors = (
    df.select(
        "contributor_key",
        "contributor_type",
        "contributor_name",
        "contributor_last_name",
        "contributor_first_name",
        "contributor_middle_initial",
        "contributor_city",
        "contributor_province",
        "contributor_postal_code",
    )
    .dropDuplicates(["contributor_key"])
)
df_contributors.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLES["contributors"])
rc = spark.table(TABLES["contributors"]).count()
print(f"  contributors: {rc:,} rows")

# ── electoral_events ────────────────────────────────────────
print(f"[{datetime.now()}] Writing electoral_events...")
df_events = (
    df.select(
        "electoral_event_key",
        "electoral_event",
        "fiscal_election_date",
        "fiscal_year",
    )
    .dropDuplicates(["electoral_event_key"])
)
df_events.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLES["electoral_events"])
rc = spark.table(TABLES["electoral_events"]).count()
print(f"  electoral_events: {rc:,} rows")

# ── contributions ───────────────────────────────────────────
print(f"[{datetime.now()}] Writing contributions...")
df_contributions = (
    df.select(
        F.monotonically_increasing_id().alias("contribution_id"),
        "recipient_id",
        "contributor_key",
        "electoral_event_key",
        "form_id",
        "financial_report",
        "part_number_of_return",
        "financial_report_part",
        "contribution_received_date",
        "monetary_amount",
        "non_monetary_amount",
        "total_amount",
        "leadership_contestant",
        "_source_file",
        "_ingested_at",
        "_transformed_at",
    )
)
df_contributions.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLES["contributions"])
rc = spark.table(TABLES["contributions"]).count()
print(f"  contributions: {rc:,} rows")

# ============================================================
# STEP 8: LIQUID CLUSTERING + PREDICTIVE OPTIMIZATION
# ============================================================
print(f"[{datetime.now()}] Configuring tables...")
spark.sql(f"ALTER TABLE {TABLES['contributions']} CLUSTER BY (contribution_received_date)")

for name, table in TABLES.items():
    spark.sql(f"ALTER TABLE {table} ENABLE PREDICTIVE OPTIMIZATION")
    print(f"  {name}: predictive optimization enabled")

# ============================================================
# STEP 9: VALIDATION
# ============================================================
print(f"\n{'='*60}")
print(f"PHASE 2 COMPLETE — Silver Layer Summary")
print(f"{'='*60}")
for name, table in TABLES.items():
    count = spark.table(table).count()
    print(f"  {table}: {count:,} rows")
print(f"  Timestamp: {datetime.now()}")
print(f"{'='*60}")

[2026-06-01 02:55:59.841474] Reading Bronze...
  Bronze rows: 284,136
  After dedup:  282,098  (dropped 2,038)
[2026-06-01 02:56:01.174007] Encoding fixes applied.
[2026-06-01 02:56:01.179911] Cleaning and type casting complete.
[2026-06-01 02:56:01.179971] Writing recipients...
  recipients: 1,004 rows
[2026-06-01 02:56:04.259598] Writing contributors...
  contributors: 123,451 rows
[2026-06-01 02:56:07.570664] Writing electoral_events...
  electoral_events: 44 rows
[2026-06-01 02:56:10.355433] Writing contributions...
  contributions: 282,098 rows
[2026-06-01 02:56:14.307620] Configuring tables...
  recipients: predictive optimization enabled
  contributors: predictive optimization enabled
  electoral_events: predictive optimization enabled
  contributions: predictive optimization enabled

PHASE 2 COMPLETE — Silver Layer Summary
  pfin_dev.silver.recipients: 1,004 rows
  pfin_dev.silver.contributors: 123,451 rows
  pfin_dev.silver.electoral_events: 44 rows
  pfin_dev.silver.contribut